# LeNet

In [6]:
import torch.nn as nn
import torch

In [7]:
class LeNet(nn.Module): # по дефолту аргуенты для LeNet-5
    def __init__(self, 
                 input_size=(32, 32), 
                 in_channels=1, 
                 output_size=10, 
                 conv_layers=(6, 16, 120),
                 activation=nn.Tanh,
                 fc_layers=(84,), 
                 conv_kernels=(5, 5, 5),
                 conv_strides=(1, 1, 1),
                 conv_paddings=(0, 0, 0),
                 pooling='avg',
                 pooling_kernel=2,
                 pooling_stride=2,
                 pool_after=(0, 1)):
        
        super().__init__()

        modules = []

        h, w = input_size

        in_ = in_channels
        for i in range(len(conv_layers)):
            modules.append(nn.Conv2d(
                in_channels=in_,
                out_channels=conv_layers[i],
                kernel_size=conv_kernels[i],
                stride=conv_strides[i],
                padding=conv_paddings[i],
                ))
            
            h = ( (h + (2 * conv_paddings[i]) - conv_kernels[i]) // conv_strides[i] ) + 1
            w = ( (w + (2 * conv_paddings[i]) - conv_kernels[i]) // conv_strides[i] ) + 1

            if i < len(conv_layers) - 1:
                modules.append(activation())

            in_ = conv_layers[i]
            if i in pool_after:
                if pooling == 'max':
                    modules.append(nn.MaxPool2d(kernel_size=pooling_kernel, stride=pooling_stride))
                else:
                    modules.append(nn.AvgPool2d(kernel_size=pooling_kernel, stride=pooling_stride))
                h = (h - pooling_kernel) // pooling_stride + 1
                w = (w - pooling_kernel) // pooling_stride + 1

        modules.append(nn.Flatten())
        
        in_ = conv_layers[-1] * h * w
        for i in range(len(fc_layers)):
            modules.append(nn.Linear(in_features=in_, out_features=fc_layers[i]))
            in_ = fc_layers[i]
            modules.append(activation())

        modules.append(nn.Linear(in_, output_size))

        self.network = nn.Sequential(*modules)

    def forward(self, x):
        return self.network(x)

In [8]:
lenet5 = LeNet()

In [10]:
x = torch.randn(4, 1, 32, 32)
y = lenet5(x)

In [11]:
y

tensor([[-0.0914,  0.1104,  0.0104, -0.0094, -0.1145,  0.0118, -0.0570, -0.0579,
         -0.0246,  0.0509],
        [-0.0381,  0.1025, -0.0122, -0.0134, -0.0941,  0.0147, -0.0731, -0.0386,
         -0.0586,  0.0462],
        [-0.0520,  0.0875, -0.0026,  0.0077, -0.0903,  0.0008, -0.0638, -0.0463,
         -0.0510,  0.0693],
        [-0.0480,  0.0952,  0.0092, -0.0200, -0.0862, -0.0002, -0.0827, -0.0266,
         -0.0521,  0.0619]], grad_fn=<AddmmBackward0>)